# Playbook 2 · The batch pipeline

**Stage:** one documented command for a processing date. Stable reruns. No
silent row loss. Evidence in automation.


> **Playbook, not walkthrough.** [`exploration.ipynb`](exploration.ipynb) is the
> narrative for a reviewer: what was built and why. These five are operational —
> one per assignment stage, each answering *what does this stage guarantee* and
> *what would it take to run it in production*. They overlap deliberately on
> evidence and not at all on purpose.


## What this stage must guarantee

1. **Every input row is accounted for**, including rows not admitted to the
   output. A row may be rejected; it may not vanish.
2. **Identical inputs, rerun, produce identical state.** Not "similar" — the
   same `run_id`.
3. **A rerun is reproducible, not merely repeatable.** Running the same
   processing date *next year*, against an archive that has grown, must still
   give the answer that date would have given.

The third is stronger than the second and is the one that matters. It comes from
`processing_ts`: a visibility cutoff that hides anything published after the
processing date, whatever is on disk when the run executes.

In [1]:
from __future__ import annotations

import datetime as dt
import tempfile
import textwrap
from pathlib import Path

from forecast_spine import coverage, fixtures, gates, normalize, pipeline, seasonal_naive

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
SQL = REPO / "sql" / "asof_join.sql"

# The assignment window. Actuals for operating day D publish on D+1, so the
# processing date is one day past the last target day.
PROCESSING_DATE = dt.date(2026, 3, 24)
WINDOW_START, WINDOW_END = dt.date(2026, 2, 22), dt.date(2026, 3, 23)
PUBLICATION_START = dt.date(2026, 2, 21)

LIVE = any(RAW.glob("load_forecast/*_csv.zip"))
raw_root = RAW if LIVE else fixtures.build("pass", Path(tempfile.mkdtemp()))
if not LIVE:
    print("LIVE VINTAGES NOT FOUND -- using synthetic fixtures.\n"
          "Structure is preserved; scale and revision behaviour are not.\n")


def build():
    """Build a throwaway warehouse. Never touches data/warehouse/.

    A scratch database keeps this notebook runnable while something else holds
    the committed one -- DuckDB is single-writer, and a SQL client with an open
    connection is enough to block it.
    """
    if LIVE:
        context = pipeline.build_context(
            PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END
        )
    else:
        context = pipeline.build_context(
            fixtures.processing_date_for("pass"), raw_root, window_days=1
        )
    con = pipeline.connect(Path(tempfile.mkdtemp()) / "playbook.duckdb")
    pipeline.load(con, context)
    pipeline.build_evaluation_dataset(con, context, SQL)
    return con, context

## Run it — the ledger

In [2]:
con, context = build()

ledger = con.execute(
    "SELECT report_key, disposition, count(*) AS rows FROM source_row GROUP BY 1, 2 ORDER BY 1, 3 DESC"
).df()
raw_rows = con.execute("SELECT sum(raw_row_count) FROM source_file").fetchone()[0]
ledger_rows = con.execute("SELECT count(*) FROM source_row").fetchone()[0]

print(ledger.to_string(index=False))
print(f"\nraw data lines across all source files : {raw_rows:,}")
print(f"rows in the ledger                     : {ledger_rows:,}")
print(f"unaccounted                            : {raw_rows - ledger_rows:,}")
print()
print("Dispositions the ledger can carry:")
for disposition in ["ACCEPTED", "DUPLICATE_IDENTICAL", *sorted(normalize.QUARANTINE_DISPOSITIONS)]:
    mark = "  <- present in this run" if disposition in set(ledger.disposition) else ""
    print(f"  {disposition:28s}{mark}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   report_key disposition    rows
  actual_load    ACCEPTED     887
load_forecast    ACCEPTED 1132080

raw data lines across all source files : 1,132,967
rows in the ledger                     : 1,132,967
unaccounted                            : 0

Dispositions the ledger can carry:
  ACCEPTED                      <- present in this run
  DUPLICATE_IDENTICAL         
  QUARANTINED_INVALID_VALUE   
  QUARANTINED_KEY_CONFLICT    
  QUARANTINED_PARSE_ERROR     
  QUARANTINED_SCHEMA_ERROR    


**Only `ACCEPTED` occurs.** Across 1.13M real rows, nothing quarantined. The
quarantine branches are exercised by `forecast-spine demo` and by
`tests/test_accounting.py` — not by naturally occurring bad data.

That is honest but weaker evidence than a real bad row would be, and it is
listed as an open issue rather than glossed. In production this number is a
monitoring signal in its own right: a quarantine rate that is *always* zero
usually means the detector is broken, not that the source is clean.

## Run it — determinism

In [3]:
again = (
    pipeline.build_context(PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END)
    if LIVE else
    pipeline.build_context(context.processing_date, raw_root, window_days=1)
)
print(f"run_id, first build  {context.run_id}")
print(f"run_id, rebuilt      {again.run_id}")
print(f"identical            {context.run_id == again.run_id}")

if LIVE:
    earlier = pipeline.build_context(
        dt.date(2026, 3, 10), raw_root, window_start=WINDOW_START, window_end=dt.date(2026, 3, 9)
    )
    print(f"\nfiles visible at {context.processing_date}: {len(context.source_files):,}")
    print(f"files visible at {earlier.processing_date}: {len(earlier.source_files):,}")
    print(f"run_id differs: {context.run_id != earlier.run_id}")

print("\nrun_id = sha256(pipeline_version + processing_date + sorted content hashes)")
print("Stable for the same inputs; moves when the visible input set moves.")

run_id, first build  ae80e7096413957c6de9fafe9058c20b
run_id, rebuilt      ae80e7096413957c6de9fafe9058c20b
identical            True



files visible at 2026-03-24: 775
files visible at 2026-03-10: 449
run_id differs: True

run_id = sha256(pipeline_version + processing_date + sorted content hashes)
Stable for the same inputs; moves when the visible input set moves.


## What breaks

| Failure | How it presents | Caught by |
| --- | --- | --- |
| Loader drops rows on a parse path | Output looks fine; totals are quietly short | `SOURCE_ROWS_UNACCOUNTED` — the gate re-counts from `source_file.raw_row_count` and does not trust the loader |
| Source header changes | New columns, or reordered ones | `schema_fingerprint` → `SCHEMA_DRIFT` |
| Zone columns mapped by position | **Complete, plausible, wrong** data | Nothing automatic. NP3-565 and NP6-345 order `SOUTHERN`/`SOUTH_CENTRAL` differently — the map is by name, by construction |
| Two processes write at once | `IO Error: Conflicting lock` | DuckDB is single-writer. A SQL client with an open connection is enough |
| Partial load after a crash | Mixed state in the warehouse | `run_id` is the natural key; re-running replaces cleanly |

## In production

**Orchestration.** One DAG, one task per stage, `processing_date` as the
partition key. Airflow's `logical_date` maps exactly onto `--processing-date`,
which is not a coincidence — the CLI was shaped so a scheduler would not need a
translation layer.

**Idempotency.** `run_id` is the natural key. A task that finds its `run_id`
already complete is a no-op. This makes retries free and makes "just run it
again" a safe instruction rather than a hopeful one.

**Backfill is the same DAG over a date range.** Because `processing_ts` gates
visibility, backfilling six months produces the answers those dates *would have*
produced — not today's answers wearing old dates. Very few pipelines have this
property, and it is worth stating explicitly to whoever inherits it.

**Storage.** DuckDB is the right call for a reviewer and the wrong one for a
deployment: single-writer, no replication, no point-in-time recovery. The
migration order I would use:

| Scale | Store | Why |
| --- | --- | --- |
| One month, one operator | DuckDB file | Rebuilds in 42.6s; no infrastructure |
| Multi-user, < 1TB | Postgres | Concurrency, backups, familiar ops |
| Multi-year vintages | Iceberg or Delta on object storage | `forecast_vintage` grows ~12.6M rows/month; partition by target date |

The as-of query is plain SQL precisely so this migration is a port, not a
rewrite.

**`source_row` is the expensive table** — 1.13M rows per month here, and it
stores every raw line verbatim. That is the price of row accountability. In
production: keep full fidelity hot for ~90 days, then compact to
quarantined-rows-only plus per-file counts. Never drop the counts; they are what
makes the accounting check possible at all.

**What to cache on a full rebuild.** Normalization is the expensive step, not
retrieval. Cache normalized rows keyed by `content_sha256`, so a rebuild that
re-reads the same immutable files skips parsing entirely.

## Runbook

| Task | Command | Notes |
| --- | --- | --- |
| Run one processing date | `forecast-spine run --processing-date D --window-start A --window-end B` | Exit 0 = both gates passed |
| Reprocess a date | The same command, unchanged | Idempotent; same inputs → same `run_id` |
| Backfill a range | Loop the same command per date | Each date independent; safe to parallelise only with separate storage |
| Verify no row loss | Readiness gate | Fails on a **one-row** disagreement |
| "Is this output stale?" | Compare `run_id` to a rebuild | Different `run_id` → inputs changed |
| Lock contention | `IO Error: Conflicting lock` | Close the SQL client. Connect read-only next time. |

## Where this lives

`src/forecast_spine/pipeline.py` · `normalize.py` · `cli.py` ·
`tests/test_rerun.py` · `tests/test_accounting.py`